# به نام خدا

### تشخیص سه مربع و تشخیص درست جهت

In [ ]:
import cv2
import numpy as np
import math

VIDEO_PATH = r"D:\FiraCUP\Soraat\PishVaDor\road.mp4"
DISPLAY_SCALE = 0.7

# ===== حافظه =====
prev_boxes_x = [None, None, None]  # چپ، وسط، راست

ALPHA = 0.25
MAX_JUMP = 80

# ================= توابع کمکی =================
def draw_square(img, center, size=14, color=(0,255,255)):
    x, y = center
    cv2.rectangle(img, (x-size, y-size),
                  (x+size, y+size), color, 2)

def smooth(prev, new):
    if prev is None:
        return new
    if abs(new - prev) > MAX_JUMP:
        return prev
    return int(ALPHA * new + (1 - ALPHA) * prev)

# ================= پردازش فریم =================
def process_frame(frame):
    global prev_boxes_x

    h, w = frame.shape[:2]
    car_x = w // 2
    y_ref = int(h * 0.72)

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)
    edges = cv2.Canny(blur, 50, 150)

    mask = np.zeros_like(edges)
    roi = np.array([[ (0,h),(w,h),(w,int(h*0.6)),(0,int(h*0.6)) ]])
    cv2.fillPoly(mask, roi, 255)
    edges = cv2.bitwise_and(edges, mask)

    lines = cv2.HoughLinesP(
        edges, 1, np.pi/180,
        threshold=50,
        minLineLength=50,
        maxLineGap=140
    )

    xs = []

    # ===== جمع‌آوری نقاط X خطوط =====
    if lines is not None:
        for l in lines:
            x1,y1,x2,y2 = l[0]
            if abs(x2-x1) < 10:
                continue

            slope = (y2-y1)/(x2-x1)
            if abs(slope) < 0.3 or abs(slope) > 2:
                continue

            x_at_y = int(x1 + (y_ref-y1)*(x2-x1)/(y2-y1))
            if 0 < x_at_y < w:
                xs.append(x_at_y)

            cv2.line(frame,(x1,y1),(x2,y2),(100,100,255),1)

    if len(xs) < 10:
        return frame

    # ===== هیستوگرام تراکم =====
    hist = np.histogram(xs, bins=60, range=(0,w))[0]
    peaks = np.argsort(hist)[-3:]  # سه پیک بزرگ

    peaks = sorted(peaks)
    boxes_x = []

    for i, p in enumerate(peaks):
        x = int((p + 0.5) * (w / 60))
        x = smooth(prev_boxes_x[i], x)
        boxes_x.append(x)

    prev_boxes_x = boxes_x

    # ===== رسم سه مربع =====
    colors = [(255,0,0),(0,255,255),(0,255,0)]
    for i, x in enumerate(boxes_x):
        draw_square(frame, (x, y_ref), color=colors[i])

    # ===== تشخیص لاین =====
    center_x = boxes_x[1]
    lane = "راست" if center_x < car_x else "چپ"

    # ===== انتخاب دو مربع فعال =====
    if lane == "راست":
        active = (boxes_x[1], boxes_x[2])
    else:
        active = (boxes_x[0], boxes_x[1])

    steer_x = (active[0] + active[1]) // 2

    # ===== فلش =====
    cv2.arrowedLine(frame,
                    (car_x, h),
                    (steer_x, y_ref),
                    (0,255,0), 3)

    cv2.putText(frame,
        f"لاین: {lane}",
        (40,50),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,(0,255,0),2)

    return frame

# ================= اجرای ویدیو =================
cap = cv2.VideoCapture(VIDEO_PATH)
print("🎥 Lane Density Tracking (q خروج)")

while True:
    ret, frame = cap.read()
    if not ret:
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        continue

    processed = process_frame(frame)
    display = cv2.resize(processed, None,
                          fx=DISPLAY_SCALE,
                          fy=DISPLAY_SCALE)

    cv2.imshow("Stable Lane Squares", display)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


🎥 Lane Density Tracking (q خروج)


KeyboardInterrupt: 

: 

### کد اصلی  تا اینجا

In [13]:
import cv2
import numpy as np
import math

VIDEO_PATH = r"D:\FiraCUP\Soraat\PishVaDor\road.mp4"
DISPLAY_SCALE = 0.7

# ===== حافظه =====
prev_boxes_x = [None, None, None]  # چپ، وسط، راست

ALPHA = 0.25
MAX_JUMP = 80

# ================= توابع کمکی =================
def draw_square(img, center, size=14, color=(0,255,255)):
    x, y = center
    cv2.rectangle(img, (x-size, y-size),
                  (x+size, y+size), color, 2)

def smooth(prev, new):
    if prev is None:
        return new
    if abs(new - prev) > MAX_JUMP:
        return prev
    return int(ALPHA * new + (1 - ALPHA) * prev)

# ===== محاسبه زاویه نسبت به خط عمودی =====
def angle_between_vertical(p1, p2):
    dx = p2[0] - p1[0]
    dy = p1[1] - p2[1]   # چون محور y برعکس است
    angle = math.degrees(math.atan2(dx, dy))
    return angle

# ================= پردازش فریم =================
def process_frame(frame):
    global prev_boxes_x

    h, w = frame.shape[:2]
    car_x = w // 2
    y_ref = int(h * 0.72)

    car_point = (car_x, h)
    vertical_ref = (car_x, int(h * 0.6))

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)
    edges = cv2.Canny(blur, 50, 150)

    mask = np.zeros_like(edges)
    roi = np.array([[ (0,h),(w,h),(w,int(h*0.6)),(0,int(h*0.6)) ]])
    cv2.fillPoly(mask, roi, 255)
    edges = cv2.bitwise_and(edges, mask)

    lines = cv2.HoughLinesP(
        edges, 1, np.pi/180,
        threshold=50,
        minLineLength=50,
        maxLineGap=140
    )

    xs = []

    # ===== جمع‌آوری نقاط X خطوط =====
    if lines is not None:
        for l in lines:
            x1,y1,x2,y2 = l[0]
            if abs(x2-x1) < 10:
                continue

            slope = (y2-y1)/(x2-x1)
            if abs(slope) < 0.3 or abs(slope) > 2:
                continue

            x_at_y = int(x1 + (y_ref-y1)*(x2-x1)/(y2-y1))
            if 0 < x_at_y < w:
                xs.append(x_at_y)

            cv2.line(frame,(x1,y1),(x2,y2),(100,100,255),1)

    if len(xs) < 10:
        return frame

    # ===== هیستوگرام تراکم =====
    hist = np.histogram(xs, bins=60, range=(0,w))[0]
    peaks = np.argsort(hist)[-3:]
    peaks = sorted(peaks)

    boxes_x = []
    for i, p in enumerate(peaks):
        x = int((p + 0.5) * (w / 60))
        x = smooth(prev_boxes_x[i], x)
        boxes_x.append(x)

    prev_boxes_x = boxes_x

    # ===== رسم سه مربع =====
    colors = [(255,0,0),(0,255,255),(0,255,0)]
    for i, x in enumerate(boxes_x):
        draw_square(frame, (x, y_ref), color=colors[i])

    # ===== تشخیص لاین طبق قانون شما =====
    left_count = sum(1 for x in boxes_x if x < car_x)
    right_count = sum(1 for x in boxes_x if x > car_x)

    if left_count == 2 and right_count == 1:
        lane = "راست"
        active = (boxes_x[1], boxes_x[2])
    elif right_count == 2 and left_count == 1:
        lane = "چپ"
        active = (boxes_x[0], boxes_x[1])
    else:
        lane = "نامشخص"
        active = (boxes_x[0], boxes_x[2])

    steer_x = (active[0] + active[1]) // 2
    steer_point = (steer_x, y_ref)

    # ===== فلش =====
    cv2.arrowedLine(frame,
                    car_point,
                    steer_point,
                    (0,255,0), 3)

    # ===== خط عمودی مرجع =====
    cv2.line(frame,
             car_point,
             vertical_ref,
             (255,255,255), 2)

    # ===== زاویه =====
    angle = angle_between_vertical(car_point, steer_point)

    # ===== نمایش =====
    cv2.putText(frame,
        f"Lane: {lane}",
        (40,50),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,(0,255,0),2)

    cv2.putText(frame,
        f"Angle: {angle:+.1f}",
        (40,90),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9,(255,255,255),2)

    # ===== چاپ در ترمینال =====
    print(f"Angle: {angle:+.2f} deg | Lane: {lane}")

    return frame

# ================= اجرای ویدیو =================
cap = cv2.VideoCapture(VIDEO_PATH)
print("🎥 Lane Density Tracking (q خروج)")

while True:
    ret, frame = cap.read()
    if not ret:
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        continue

    processed = process_frame(frame)
    display = cv2.resize(processed, None,
                          fx=DISPLAY_SCALE,
                          fy=DISPLAY_SCALE)

    cv2.imshow("Stable Lane Squares", display)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


🎥 Lane Density Tracking (q خروج)
Angle: +6.03 deg | Lane: راست
Angle: +6.78 deg | Lane: راست
Angle: +7.33 deg | Lane: راست
Angle: +7.71 deg | Lane: راست
Angle: +7.89 deg | Lane: راست
Angle: +8.08 deg | Lane: راست
Angle: +8.26 deg | Lane: راست
Angle: +8.45 deg | Lane: راست
Angle: +8.45 deg | Lane: راست
Angle: +9.37 deg | Lane: راست
Angle: +8.63 deg | Lane: راست
Angle: +8.82 deg | Lane: راست
Angle: +9.55 deg | Lane: راست
Angle: +10.11 deg | Lane: راست
Angle: +10.47 deg | Lane: راست
Angle: +10.47 deg | Lane: راست
Angle: +10.65 deg | Lane: راست
Angle: +10.84 deg | Lane: راست
Angle: +10.29 deg | Lane: راست
Angle: +10.47 deg | Lane: راست
Angle: +10.11 deg | Lane: راست
Angle: +10.11 deg | Lane: راست
Angle: +10.11 deg | Lane: راست
Angle: +10.11 deg | Lane: راست
Angle: +10.11 deg | Lane: راست
Angle: +10.47 deg | Lane: راست
Angle: +10.65 deg | Lane: راست
Angle: +10.84 deg | Lane: راست
Angle: +11.02 deg | Lane: راست
Angle: +11.20 deg | Lane: راست
Angle: +11.20 deg | Lane: راست
Angle: +11.38 deg |

KeyboardInterrupt: 

### تشخیص مانع

In [ ]:
import cv2
import numpy as np

CHANGE_LANE_THRESHOLD = 450

current_lane = "center"
original_lane = "center"
lane_changed = False


def process_frame(frame, lane, boxes_x, y_ref):
    global current_lane, original_lane, lane_changed

    output = frame.copy()

    # ================== 1) تشخیص مانع (عین کد خودت) ==================
    orange_mask = detect_orange_obstacles(frame)
    contour, area = find_largest_obstacle(orange_mask)
    obstacle_detected = area >= CHANGE_LANE_THRESHOLD

    obstacle_in_our_lane = False
    cx = None

    if contour is not None:
        cv2.drawContours(output, [contour], -1, (0,255,0), 2)

        M = cv2.moments(contour)
        if M["m00"] != 0:
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            cv2.circle(output, (cx, cy), 5, (255,0,0), -1)
            cv2.putText(output, f"Area:{int(area)}", (cx-40, cy-20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)

    # ================== 2) تشخیص اینکه مانع در لاین ماست ==================
    if obstacle_detected and cx is not None:
        frame_center = frame.shape[1] // 2

        if current_lane == "left" and cx < frame_center:
            obstacle_in_our_lane = True
        elif current_lane == "right" and cx > frame_center:
            obstacle_in_our_lane = True

    # ================== 3) تصمیم تغییر لاین ==================
    steer_x = None

    if obstacle_detected and obstacle_in_our_lane and not lane_changed:
        # ثبت لاین اولیه
        original_lane = current_lane

        # تغییر لاین
        if current_lane == "left":
            current_lane = "right"
            steer_x = (boxes_x[1] + boxes_x[2]) // 2
        else:
            current_lane = "left"
            steer_x = (boxes_x[0] + boxes_x[1]) // 2

        lane_changed = True

    # ================== 4) بازگشت به لاین اصلی ==================
    elif lane_changed and not obstacle_detected:
        current_lane = original_lane
        lane_changed = False

        if current_lane == "left":
            steer_x = (boxes_x[0] + boxes_x[1]) // 2
        else:
            steer_x = (boxes_x[1] + boxes_x[2]) // 2

    # ================== 5) حالت عادی (بدون مانع) ==================
    if steer_x is None:
        if current_lane == "left":
            steer_x = (boxes_x[0] + boxes_x[1]) // 2
        else:
            steer_x = (boxes_x[1] + boxes_x[2]) // 2

    steer_point = (steer_x, y_ref)

    # ================== 6) رسم فلش ==================
    cv2.arrowedLine(
        output,
        (frame.shape[1] // 2, frame.shape[0]),
        steer_point,
        (0,0,255),
        4,
        tipLength=0.3
    )

    cv2.putText(output, f"Lane: {current_lane}", (10,30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)

    return output


Error: Cannot open video file


### کد نهایی با ورودی فیلم

In [20]:
import cv2
import numpy as np
import math

VIDEO_PATH = r"D:\FiraCUP\Soraat\PishVaDor\20260105-1506-22.6829242.mp4"
DISPLAY_SCALE = 0.9

# ===== حافظه =====
prev_boxes_x = [None, None, None]  # چپ، وسط، راست
ALPHA = 0.25
MAX_JUMP = 80

# ===== مانع =====
CHANGE_LANE_THRESHOLD = 800
original_lane = None
lane_changed = False
obstacle_frame_count = 0
OBSTACLE_CONFIRMATION_FRAMES = 5
obstacle_distance_threshold = 100

# ================= توابع کمکی =================
def draw_square(img, center, size=14, color=(0,255,255)):
    x, y = center
    cv2.rectangle(img, (x-size, y-size),
                  (x+size, y+size), color, 2)

def smooth(prev, new):
    if prev is None:
        return new
    if abs(new - prev) > MAX_JUMP:
        return prev
    return int(ALPHA * new + (1 - ALPHA) * prev)

def angle_between_vertical(p1, p2):
    """محاسبه زاویه فلش نسبت به خط عمودی (محور Y معکوس)"""
    dx = p2[0] - p1[0]
    dy = p1[1] - p2[1]  # چون محور Y در تصویر برعکس است
    
    # محاسبه زاویه به درجه (محدوده -90 تا 90)
    angle_rad = math.atan2(dx, dy)
    angle_deg = math.degrees(angle_rad)
    
    # محدود کردن زاویه به بازه -90 تا 90 درجه
    if angle_deg > 90:
        angle_deg = 90
    elif angle_deg < -90:
        angle_deg = -90
    
    return angle_deg

# ================= تشخیص مانع =================
def detect_orange_obstacles(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    lower1 = np.array([5, 150, 150])
    upper1 = np.array([15, 255, 255])
    lower2 = np.array([170, 150, 150])
    upper2 = np.array([180, 255, 255])

    mask = cv2.bitwise_or(
        cv2.inRange(hsv, lower1, upper1),
        cv2.inRange(hsv, lower2, upper2)
    )

    kernel = np.ones((7,7), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    return mask

def find_largest_obstacle(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, 0
    
    min_area = 300
    valid_contours = [c for c in contours if cv2.contourArea(c) > min_area]
    
    if not valid_contours:
        return None, 0
    
    c = max(valid_contours, key=cv2.contourArea)
    return c, cv2.contourArea(c)

def is_obstacle_in_lane(cx, cy, lane, car_x, h):
    if cy < h * 0.5:
        return False
    
    if lane == "راست":
        return cx > car_x + 50 and abs(cx - car_x) < 300
    elif lane == "چپ":
        return cx < car_x - 50 and abs(cx - car_x) < 300
    
    return False

# ================= پردازش فریم =================
def process_frame(frame):
    global prev_boxes_x, original_lane, lane_changed, obstacle_frame_count

    h, w = frame.shape[:2]
    car_x = w // 2
    y_ref = int(h * 0.72)

    car_point = (car_x, h)
    vertical_ref = (car_x, int(h * 0.6))

    # ================== 1) تشخیص لاین ==================
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)
    edges = cv2.Canny(blur, 50, 150)

    mask = np.zeros_like(edges)
    roi = np.array([[(0,h),(w,h),(w,int(h*0.6)),(0,int(h*0.6))]])
    cv2.fillPoly(mask, roi, 255)
    edges = cv2.bitwise_and(edges, mask)

    lines = cv2.HoughLinesP(
        edges, 1, np.pi/180,
        threshold=50,
        minLineLength=50,
        maxLineGap=140
    )

    xs = []

    if lines is not None:
        for l in lines:
            x1,y1,x2,y2 = l[0]
            if abs(x2-x1) < 10:
                continue

            slope = (y2-y1)/(x2-x1)
            if abs(slope) < 0.3 or abs(slope) > 2:
                continue

            x_at_y = int(x1 + (y_ref-y1)*(x2-x1)/(y2-y1))
            if 0 < x_at_y < w:
                xs.append(x_at_y)

            cv2.line(frame,(x1,y1),(x2,y2),(100,100,255),1)

    if len(xs) < 10:
        cv2.putText(frame, "خطوط کافی نیست", (40,130),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)
        
        # چاپ در ترمینال حتی وقتی خطوط کافی نیست
        print("┌─────────────────────────────────────────────────────┐")
        print("│ Angle:    ---.--°  │ Lane:         نامشخص          │")
        print("│ Lane_Changed: False │ Obstacle:       No            │")
        print("└─────────────────────────────────────────────────────┘")
        
        return frame

    # ================== 2) مربع‌ها ==================
    hist = np.histogram(xs, bins=60, range=(0,w))[0]
    peaks = sorted(np.argsort(hist)[-3:])

    boxes_x = []
    for i, p in enumerate(peaks):
        x = int((p + 0.5) * (w / 60))
        x = smooth(prev_boxes_x[i], x)
        boxes_x.append(x)

    prev_boxes_x = boxes_x

    colors = [(255,0,0),(0,255,255),(0,255,0)]
    for i, x in enumerate(boxes_x):
        draw_square(frame, (x, y_ref), color=colors[i])

    # ================== 3) تشخیص لاین فعلی ==================
    left_count = sum(1 for x in boxes_x if x < car_x)
    right_count = sum(1 for x in boxes_x if x > car_x)

    if left_count == 2 and right_count == 1:
        lane = "راست"
        active_normal = (boxes_x[1], boxes_x[2])
    elif right_count == 2 and left_count == 1:
        lane = "چپ"
        active_normal = (boxes_x[0], boxes_x[1])
    else:
        lane = "نامشخص"
        active_normal = (boxes_x[0], boxes_x[2])

    if original_lane is None and lane != "نامشخص":
        original_lane = lane

    # ================== 4) تشخیص مانع ==================
    obstacle_detected = False
    obstacle_in_our_lane = False
    cx = None
    cy = None
    area = 0

    orange_mask = detect_orange_obstacles(frame)
    contour, area = find_largest_obstacle(orange_mask)

    if contour is not None and area > 0:
        M = cv2.moments(contour)
        if M["m00"] != 0:
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            
            cv2.circle(frame, (cx, cy), 8, (0,0,255), -1)
            cv2.putText(frame, f"({cx},{cy})", (cx+10, cy), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 1)
            
            obstacle_in_our_lane = is_obstacle_in_lane(cx, cy, lane, car_x, h)
            obstacle_detected = area >= CHANGE_LANE_THRESHOLD
            
            if obstacle_detected and obstacle_in_our_lane:
                obstacle_frame_count += 1
            else:
                obstacle_frame_count = max(0, obstacle_frame_count - 1)

    # ================== 5) منطق تغییر لاین ==================
    target_lane = lane
    steer_x = None

    obstacle_confirmed = obstacle_frame_count >= OBSTACLE_CONFIRMATION_FRAMES

    if not obstacle_confirmed or not obstacle_in_our_lane:
        if lane_changed:
            if original_lane == "راست":
                steer_x = (boxes_x[1] + boxes_x[2]) // 2
                target_lane = "راست (بازگشت)"
            elif original_lane == "چپ":
                steer_x = (boxes_x[0] + boxes_x[1]) // 2
                target_lane = "چپ (بازگشت)"
            lane_changed = False
            obstacle_frame_count = 0
        else:
            if lane == "راست":
                steer_x = (boxes_x[1] + boxes_x[2]) // 2
            elif lane == "چپ":
                steer_x = (boxes_x[0] + boxes_x[1]) // 2
            else:
                steer_x = (boxes_x[0] + boxes_x[2]) // 2
            target_lane = lane

    elif obstacle_confirmed and obstacle_in_our_lane and not lane_changed:
        if original_lane is None:
            original_lane = lane
        
        if lane == "راست":
            steer_x = (boxes_x[0] + boxes_x[1]) // 2
            target_lane = "چپ (تغییر)"
            lane_changed = True
        elif lane == "چپ":
            steer_x = (boxes_x[1] + boxes_x[2]) // 2
            target_lane = "راست (تغییر)"
            lane_changed = True

    # ================== 6) محاسبه فلش و زاویه ==================
    if steer_x is None:
        if lane == "راست":
            steer_x = (boxes_x[1] + boxes_x[2]) // 2
        elif lane == "چپ":
            steer_x = (boxes_x[0] + boxes_x[1]) // 2
        else:
            steer_x = (boxes_x[0] + boxes_x[2]) // 2
    
    steer_point = (steer_x, y_ref)

    # محاسبه زاویه
    angle = angle_between_vertical(car_point, steer_point)

    # ================== 7) رسم فلش و خطوط ==================
    if lane_changed:
        arrow_color = (0,255,255)
    else:
        arrow_color = (0,255,0)
    
    cv2.arrowedLine(frame, car_point, steer_point, arrow_color, 3)
    cv2.line(frame, car_point, vertical_ref, (255,255,255), 2)

    # ================== 8) نمایش اطلاعات روی تصویر ==================
    lane_color = (0,255,0) if not lane_changed else (0,255,255)
    cv2.putText(frame,
        f"Lane: {target_lane}",
        (40,50),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9, lane_color, 2)

    cv2.putText(frame,
        f"Angle: {angle:+.1f}°",
        (40,90),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.9, (255,255,255), 2)

    if obstacle_detected:
        obstacle_status = "در لاین" if obstacle_in_our_lane else "خارج لاین"
        status_color = (0,0,255) if obstacle_in_our_lane else (0,165,255)
        cv2.putText(frame,
            f"Obstacle: {obstacle_status}",
            (40,130),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7, status_color, 2)
        
        cv2.putText(frame,
            f"Area: {int(area)}, Frames: {obstacle_frame_count}/{OBSTACLE_CONFIRMATION_FRAMES}",
            (40,160),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7, status_color, 2)
        
        if obstacle_in_our_lane and lane_changed:
            cv2.putText(frame,
                "! تغییر لاین فعال",
                (40,190),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8, (0,255,255), 2)

    if original_lane:
        cv2.putText(frame,
            f"Original: {original_lane}",
            (w-250,50),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7, (200,200,200), 2)

    # ================== 9) چاپ اطلاعات در ترمینال ==================
    # قالب‌بندی زیبا برای ترمینال
    angle_str = f"{angle:+.2f}"
    
    # فرمت‌بندی لاین
    if lane == "نامشخص":
        lane_display = "نامشخص    "
    elif lane == "راست":
        lane_display = "راست      "
    else:
        lane_display = "چپ        "
    
    # فرمت‌بندی وضعیت مانع
    obstacle_display = "Yes      " if obstacle_detected else "No       "
    
    # چاپ در ترمینال با فرمت جدول
    print("┌─────────────────────────────────────────────────────┐")
    print(f"│ Angle:    {angle_str:>7}° │ Lane:         {lane_display}│")
    print(f"│ Lane_Changed: {str(lane_changed):<5} │ Obstacle:       {obstacle_display}│")
    print("└─────────────────────────────────────────────────────┘")

    return frame

# ================= اجرای اصلی =================
cap = cv2.VideoCapture(VIDEO_PATH)
print("\n" + "="*60)
print("🎥 سیستم تشخیص لاین و مانع - نسخه نهایی")
print("="*60)
print("📊 اطلاعات خروجی در ترمینال:")
print("  - Angle: زاویه فلش نسبت به خط عمودی (-90° تا +90°)")
print("  - Lane: لاین فعلی (چپ / راست / نامشخص)")
print("  - Lane_Changed: وضعیت تغییر لاین (True/False)")
print("  - Obstacle: وجود مانع در تصویر (Yes/No)")
print("="*60)
print("🎮 کلیدهای کنترل:")
print("  - q: خروج")
print("  - r: ریست سیستم")
print("="*60 + "\n")

while True:
    ret, frame = cap.read()
    if not ret:
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        continue

    processed_frame = process_frame(frame)
    display_frame = cv2.resize(processed_frame, None,
                               fx=DISPLAY_SCALE,
                               fy=DISPLAY_SCALE)

    cv2.imshow("Combined Lane & Obstacle Detection", display_frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        print("\n" + "="*60)
        print("👋 برنامه خاتمه یافت")
        print("="*60)
        break
    elif key == ord('r'):
        original_lane = None
        lane_changed = False
        obstacle_frame_count = 0
        prev_boxes_x = [None, None, None]
        print("\n" + "="*60)
        print("🔄 سیستم ریست شد")
        print("="*60)

cap.release()
cv2.destroyAllWindows()


🎥 سیستم تشخیص لاین و مانع - نسخه نهایی
📊 اطلاعات خروجی در ترمینال:
  - Angle: زاویه فلش نسبت به خط عمودی (-90° تا +90°)
  - Lane: لاین فعلی (چپ / راست / نامشخص)
  - Lane_Changed: وضعیت تغییر لاین (True/False)
  - Obstacle: وجود مانع در تصویر (Yes/No)
🎮 کلیدهای کنترل:
  - q: خروج
  - r: ریست سیستم



KeyboardInterrupt: 

In [17]:
import cv2
import numpy as np
import math

VIDEO_PATH = r"D:\FiraCUP\Soraat\PishVaDor\roadd.mp4"
DISPLAY_SCALE = 0.8  # کمی کوچکتر شد تا فضای بیشتری داشته باشیم

# ===== حافظه =====
prev_boxes_x = [None, None, None]  # چپ، وسط، راست
ALPHA = 0.25
MAX_JUMP = 80

# ===== مانع =====
CHANGE_LANE_THRESHOLD = 800
original_lane = None
lane_changed = False
obstacle_frame_count = 0
OBSTACLE_CONFIRMATION_FRAMES = 5
obstacle_distance_threshold = 100

# ================= توابع کمکی =================
def draw_square(img, center, size=14, color=(0,255,255)):
    x, y = center
    cv2.rectangle(img, (x-size, y-size),
                  (x+size, y+size), color, 2)

def smooth(prev, new):
    if prev is None:
        return new
    if abs(new - prev) > MAX_JUMP:
        return prev
    return int(ALPHA * new + (1 - ALPHA) * prev)

def angle_between_vertical(p1, p2):
    """محاسبه زاویه فلش نسبت به خط عمودی (محور Y معکوس)"""
    dx = p2[0] - p1[0]
    dy = p1[1] - p2[1]  # چون محور Y در تصویر برعکس است
    
    # محاسبه زاویه به درجه (محدوده -90 تا 90)
    angle_rad = math.atan2(dx, dy)
    angle_deg = math.degrees(angle_rad)
    
    # محدود کردن زاویه به بازه -90 تا 90 درجه
    if angle_deg > 90:
        angle_deg = 90
    elif angle_deg < -90:
        angle_deg = -90
    
    return angle_deg

# ================= پردازش مرحله‌ای تصویر =================
def preprocess_image(frame):
    """پردازش مرحله‌ای تصویر برای تشخیص بهتر خطوط"""
    h, w = frame.shape[:2]
    
    # 1. سیاه و سفید کردن (Grayscale)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # 2. مات کردن (Gaussian Blur)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # 3. حذف نیمه بالای تصویر (ROI)
    roi_height = int(h * 0.5)  # فقط نیمه پایین تصویر
    roi = np.zeros_like(blurred)
    roi[h-roi_height:h, 0:w] = blurred[h-roi_height:h, 0:w]
    
    # 4. حذف نویز با morphological operations
    kernel = np.ones((3, 3), np.uint8)
    
    # Opening برای حذف نویزهای سفید کوچک
    opened = cv2.morphologyEx(roi, cv2.MORPH_OPEN, kernel)
    
    # Closing برای پر کردن حفره‌های سیاه کوچک
    closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel)
    
    # 5. تشخیص لبه‌ها (Canny)
    edges = cv2.Canny(closed, 50, 150)
    
    # 6. هایلایت خطوط روی تصویر اصلی
    highlighted = frame.copy()
    
    # ایجاد ماسک از خطوط تشخیص داده شده
    lines_mask = np.zeros_like(frame)
    lines_mask[edges > 0] = (0, 255, 0)  # رنگ سبز برای خطوط
    
    # ترکیب با تصویر اصلی با شفافیت
    alpha = 0.6  # شفافیت هایلایت
    highlighted = cv2.addWeighted(lines_mask, alpha, highlighted, 1 - alpha, 0)
    
    return {
        'original': frame.copy(),  # تصویر اصلی بدون تغییرات
        'gray': cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR),
        'blurred': cv2.cvtColor(blurred, cv2.COLOR_GRAY2BGR),
        'roi': cv2.cvtColor(roi, cv2.COLOR_GRAY2BGR),
        'denoised': cv2.cvtColor(closed, cv2.COLOR_GRAY2BGR),
        'edges': cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR),
        'highlighted': highlighted,
        'final_edges': edges  # برای استفاده در پردازش اصلی
    }

# ================= تشخیص مانع =================
def detect_orange_obstacles(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    lower1 = np.array([5, 150, 150])
    upper1 = np.array([15, 255, 255])
    lower2 = np.array([170, 150, 150])
    upper2 = np.array([180, 255, 255])

    mask = cv2.bitwise_or(
        cv2.inRange(hsv, lower1, upper1),
        cv2.inRange(hsv, lower2, upper2)
    )

    kernel = np.ones((7,7), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    return mask

def find_largest_obstacle(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, 0
    
    min_area = 300
    valid_contours = [c for c in contours if cv2.contourArea(c) > min_area]
    
    if not valid_contours:
        return None, 0
    
    c = max(valid_contours, key=cv2.contourArea)
    return c, cv2.contourArea(c)

def is_obstacle_in_lane(cx, cy, lane, car_x, h):
    if cy < h * 0.5:
        return False
    
    if lane == "راست":
        return cx > car_x + 50 and abs(cx - car_x) < 300
    elif lane == "چپ":
        return cx < car_x - 50 and abs(cx - car_x) < 300
    
    return False

# ================= پردازش فریم اصلی =================
def process_frame(frame):
    global prev_boxes_x, original_lane, lane_changed, obstacle_frame_count

    h, w = frame.shape[:2]
    car_x = w // 2
    y_ref = int(h * 0.72)

    car_point = (car_x, h)
    vertical_ref = (car_x, int(h * 0.6))

    # ================== 1) پردازش مرحله‌ای تصویر ==================
    # ذخیره تصویر اصلی برای نمایش در داشبورد
    original_frame_copy = frame.copy()
    
    processed_images = preprocess_image(frame)
    edges = processed_images['final_edges']
    
    # ================== 2) پردازش خطوط ==================
    # اعمال ROI روی لبه‌های پردازش شده
    mask = np.zeros_like(edges)
    roi_polygon = np.array([[(0,h),(w,h),(w,int(h*0.6)),(0,int(h*0.6))]])
    cv2.fillPoly(mask, roi_polygon, 255)
    edges_roi = cv2.bitwise_and(edges, mask)

    # تشخیص خطوط با HoughLinesP
    lines = cv2.HoughLinesP(
        edges_roi, 1, np.pi/180,
        threshold=50,
        minLineLength=50,
        maxLineGap=140
    )

    xs = []
    frame_with_lines = frame.copy()

    if lines is not None:
        for l in lines:
            x1,y1,x2,y2 = l[0]
            if abs(x2-x1) < 10:
                continue

            slope = (y2-y1)/(x2-x1)
            if abs(slope) < 0.3 or abs(slope) > 2:
                continue

            x_at_y = int(x1 + (y_ref-y1)*(x2-x1)/(y2-y1))
            if 0 < x_at_y < w:
                xs.append(x_at_y)

            # رسم خطوط روی تصویر
            cv2.line(frame_with_lines,(x1,y1),(x2,y2),(100,100,255),2)

    # اضافه کردن خطوط به تصویر پردازش شده
    processed_images['lines'] = frame_with_lines

    if len(xs) < 10:
        cv2.putText(frame, "خطوط کافی نیست", (40,130),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)
        
        print("┌─────────────────────────────────────────────────────┐")
        print("│ Angle:    ---.--°  │ Lane:         نامشخص          │")
        print("│ Lane_Changed: False │ Obstacle:       No            │")
        print("└─────────────────────────────────────────────────────┘")
        
        return frame, processed_images

    # ================== 3) مربع‌ها ==================
    hist = np.histogram(xs, bins=60, range=(0,w))[0]
    peaks = sorted(np.argsort(hist)[-3:])

    boxes_x = []
    for i, p in enumerate(peaks):
        x = int((p + 0.5) * (w / 60))
        x = smooth(prev_boxes_x[i], x)
        boxes_x.append(x)

    prev_boxes_x = boxes_x

    # رسم مربع‌ها روی تصویر پردازش شده
    colors = [(255,0,0),(0,255,255),(0,255,0)]
    for i, x in enumerate(boxes_x):
        draw_square(frame, (x, y_ref), color=colors[i])
        draw_square(processed_images['highlighted'], (x, y_ref), color=colors[i])

    # ================== 4) تشخیص لاین فعلی ==================
    left_count = sum(1 for x in boxes_x if x < car_x)
    right_count = sum(1 for x in boxes_x if x > car_x)

    if left_count == 2 and right_count == 1:
        lane = "راست"
        active_normal = (boxes_x[1], boxes_x[2])
    elif right_count == 2 and left_count == 1:
        lane = "چپ"
        active_normal = (boxes_x[0], boxes_x[1])
    else:
        lane = "نامشخص"
        active_normal = (boxes_x[0], boxes_x[2])

    if original_lane is None and lane != "نامشخص":
        original_lane = lane

    # ================== 5) تشخیص مانع ==================
    obstacle_detected = False
    obstacle_in_our_lane = False
    cx = None
    cy = None
    area = 0

    orange_mask = detect_orange_obstacles(frame)
    contour, area = find_largest_obstacle(orange_mask)

    if contour is not None and area > 0:
        M = cv2.moments(contour)
        if M["m00"] != 0:
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            
            cv2.circle(frame, (cx, cy), 8, (0,0,255), -1)
            cv2.putText(frame, f"({cx},{cy})", (cx+10, cy), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 1)
            
            obstacle_in_our_lane = is_obstacle_in_lane(cx, cy, lane, car_x, h)
            obstacle_detected = area >= CHANGE_LANE_THRESHOLD
            
            if obstacle_detected and obstacle_in_our_lane:
                obstacle_frame_count += 1
            else:
                obstacle_frame_count = max(0, obstacle_frame_count - 1)

    # ================== 6) منطق تغییر لاین ==================
    target_lane = lane
    steer_x = None

    obstacle_confirmed = obstacle_frame_count >= OBSTACLE_CONFIRMATION_FRAMES

    if not obstacle_confirmed or not obstacle_in_our_lane:
        if lane_changed:
            if original_lane == "راست":
                steer_x = (boxes_x[1] + boxes_x[2]) // 2
                target_lane = "راست (بازگشت)"
            elif original_lane == "چپ":
                steer_x = (boxes_x[0] + boxes_x[1]) // 2
                target_lane = "چپ (بازگشت)"
            lane_changed = False
            obstacle_frame_count = 0
        else:
            if lane == "راست":
                steer_x = (boxes_x[1] + boxes_x[2]) // 2
            elif lane == "چپ":
                steer_x = (boxes_x[0] + boxes_x[1]) // 2
            else:
                steer_x = (boxes_x[0] + boxes_x[2]) // 2
            target_lane = lane

    elif obstacle_confirmed and obstacle_in_our_lane and not lane_changed:
        if original_lane is None:
            original_lane = lane
        
        if lane == "راست":
            steer_x = (boxes_x[0] + boxes_x[1]) // 2
            target_lane = "چپ (تغییر)"
            lane_changed = True
        elif lane == "چپ":
            steer_x = (boxes_x[1] + boxes_x[2]) // 2
            target_lane = "راست (تغییر)"
            lane_changed = True

    # ================== 7) محاسبه فلش و زاویه ==================
    if steer_x is None:
        if lane == "راست":
            steer_x = (boxes_x[1] + boxes_x[2]) // 2
        elif lane == "چپ":
            steer_x = (boxes_x[0] + boxes_x[1]) // 2
        else:
            steer_x = (boxes_x[0] + boxes_x[2]) // 2
    
    steer_point = (steer_x, y_ref)

    # محاسبه زاویه
    angle = angle_between_vertical(car_point, steer_point)

    # ================== 8) رسم فلش و خطوط ==================
    if lane_changed:
        arrow_color = (0,255,255)
    else:
        arrow_color = (0,255,0)
    
    # رسم روی تصویر اصلی
    cv2.arrowedLine(frame, car_point, steer_point, arrow_color, 3)
    cv2.line(frame, car_point, vertical_ref, (255,255,255), 2)
    
    # رسم روی تصویر هایلایت شده
    cv2.arrowedLine(processed_images['highlighted'], car_point, steer_point, arrow_color, 3)
    cv2.line(processed_images['highlighted'], car_point, vertical_ref, (255,255,255), 2)

    # ================== 9) نمایش اطلاعات روی تصویر ==================
    lane_color = (0,255,0) if not lane_changed else (0,255,255)
    
    # روی تصویر اصلی
    cv2.putText(frame, f"Lane: {target_lane}", (40,50),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, lane_color, 2)
    cv2.putText(frame, f"Angle: {angle:+.1f}°", (40,90),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)
    
    # روی تصویر هایلایت شده
    cv2.putText(processed_images['highlighted'], f"Lane: {target_lane}", (40,50),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, lane_color, 2)
    cv2.putText(processed_images['highlighted'], f"Angle: {angle:+.1f}°", (40,90),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)

    if obstacle_detected:
        obstacle_status = "در لاین" if obstacle_in_our_lane else "خارج لاین"
        status_color = (0,0,255) if obstacle_in_our_lane else (0,165,255)
        
        cv2.putText(frame, f"Obstacle: {obstacle_status}", (40,130),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, status_color, 2)
        cv2.putText(frame, f"Area: {int(area)}", (40,160),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, status_color, 2)
        
        cv2.putText(processed_images['highlighted'], f"Obstacle: {obstacle_status}", (40,130),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, status_color, 2)

        if obstacle_in_our_lane and lane_changed:
            cv2.putText(frame, "! تغییر لاین فعال", (40,190),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)
            cv2.putText(processed_images['highlighted'], "! تغییر لاین فعال", (40,190),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)

    if original_lane:
        cv2.putText(frame, f"Original: {original_lane}", (w-250,50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200,200,200), 2)

    # ================== 10) چاپ اطلاعات در ترمینال ==================
    angle_str = f"{angle:+.2f}"
    
    if lane == "نامشخص":
        lane_display = "نامشخص    "
    elif lane == "راست":
        lane_display = "راست      "
    else:
        lane_display = "چپ        "
    
    obstacle_display = "Yes      " if obstacle_detected else "No       "
    
    print("┌─────────────────────────────────────────────────────┐")
    print(f"│ Angle:    {angle_str:>7}° │ Lane:         {lane_display}│")
    print(f"│ Lane_Changed: {str(lane_changed):<5} │ Obstacle:       {obstacle_display}│")
    print("└─────────────────────────────────────────────────────┘")

    return frame, processed_images

def create_dashboard(processed_frame, processed_images):
    """ایجاد داشبورد با تمام مراحل پردازش"""
    h, w = processed_frame.shape[:2]
    
    # محاسبه اندازه برای نمایش
    small_h, small_w = h // 3, w // 3
    
    # کوچک کردن تصاویر
    resized_images = {}
    for key, img in processed_images.items():
        if key != 'final_edges':  # edges نهایی را جداگانه پردازش می‌کنیم
            resized_images[key] = cv2.resize(img, (small_w, small_h))
    
    # ایجاد شبکه 3x3 برای نمایش
    dashboard = np.zeros((small_h * 3, small_w * 3, 3), dtype=np.uint8)
    
    # مرتب کردن تصاویر در داشبورد
    positions = [
        ('original', 0, 0, "Original (Input)"),
        ('gray', 0, 1, "Grayscale"),
        ('blurred', 0, 2, "Blurred"),
        ('roi', 1, 0, "ROI (Lower Half)"),
        ('denoised', 1, 1, "Denoised"),
        ('edges', 1, 2, "Edges (Canny)"),
        ('lines', 2, 0, "Detected Lines"),
        ('highlighted', 2, 1, "Highlighted + Squares"),
        ('result', 2, 2, "Final Result")
    ]
    
    # اضافه کردن فریم نهایی به processed_images
    processed_images['result'] = processed_frame
    resized_images['result'] = cv2.resize(processed_frame, (small_w, small_h))
    
    for key, row, col, title in positions:
        if key in resized_images:
            # اضافه کردن عنوان
            cv2.putText(resized_images[key], title, (10, 25), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
            
            # اضافه کردن کادر
            cv2.rectangle(resized_images[key], (0, 0), 
                         (small_w-1, small_h-1), (100, 100, 100), 1)
            
            # قرار دادن در داشبورد
            y_start = row * small_h
            y_end = y_start + small_h
            x_start = col * small_w
            x_end = x_start + small_w
            dashboard[y_start:y_end, x_start:x_end] = resized_images[key]
    
    return dashboard

# ================= اجرای اصلی =================
cap = cv2.VideoCapture(VIDEO_PATH)
print("\n" + "="*60)
print("🚗 سیستم تشخیص لاین و مانع - نسخه پردازش مرحله‌ای")
print("="*60)
print("🔬 مراحل پردازش:")
print("  1. تبدیل به خاکستری")
print("  2. مات کردن (Blur)")
print("  3. حذف نیمه بالای تصویر")
print("  4. حذف نویز")
print("  5. تشخیص لبه‌ها")
print("  6. هایلایت خطوط")
print("  7. تشخیص مربع‌ها و لاین")
print("="*60)
print("🎮 کلیدهای کنترل:")
print("  - q: خروج")
print("  - r: ریست سیستم")
print("  - d: نمایش/عدم نمایش داشبورد")
print("="*60 + "\n")

show_dashboard = True

while True:
    ret, frame = cap.read()
    if not ret:
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        continue

    # پردازش فریم
    processed_frame, processed_images = process_frame(frame)
    
    # ایجاد داشبورد
    if show_dashboard:
        dashboard = create_dashboard(processed_frame, processed_images)
        display_frame = cv2.resize(dashboard, None, 
                                  fx=DISPLAY_SCALE, 
                                  fy=DISPLAY_SCALE)
        window_name = "🚗 پردازش مرحله‌ای تشخیص لاین - داشبورد"
    else:
        display_frame = cv2.resize(processed_frame, None,
                                  fx=DISPLAY_SCALE,
                                  fy=DISPLAY_SCALE)
        window_name = "🚗 تشخیص لاین و مانع - نمایش نهایی"

    cv2.imshow(window_name, display_frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        print("\n" + "="*60)
        print("👋 برنامه خاتمه یافت")
        print("="*60)
        break
    elif key == ord('r'):
        original_lane = None
        lane_changed = False
        obstacle_frame_count = 0
        prev_boxes_x = [None, None, None]
        print("\n" + "="*60)
        print("🔄 سیستم ریست شد")
        print("="*60)
    elif key == ord('d'):
        show_dashboard = not show_dashboard
        print(f"\n📊 نمایش داشبورد: {'فعال' if show_dashboard else 'غیرفعال'}")

cap.release()
cv2.destroyAllWindows()


🚗 سیستم تشخیص لاین و مانع - نسخه پردازش مرحله‌ای
🔬 مراحل پردازش:
  1. تبدیل به خاکستری
  2. مات کردن (Blur)
  3. حذف نیمه بالای تصویر
  4. حذف نویز
  5. تشخیص لبه‌ها
  6. هایلایت خطوط
  7. تشخیص مربع‌ها و لاین
🎮 کلیدهای کنترل:
  - q: خروج
  - r: ریست سیستم
  - d: نمایش/عدم نمایش داشبورد

┌─────────────────────────────────────────────────────┐
│ Angle:      +6.03° │ Lane:         راست      │
│ Lane_Changed: False │ Obstacle:       No       │
└─────────────────────────────────────────────────────┘
┌─────────────────────────────────────────────────────┐
│ Angle:      +6.78° │ Lane:         راست      │
│ Lane_Changed: False │ Obstacle:       No       │
└─────────────────────────────────────────────────────┘
┌─────────────────────────────────────────────────────┐
│ Angle:      +7.33° │ Lane:         راست      │
│ Lane_Changed: False │ Obstacle:       No       │
└─────────────────────────────────────────────────────┘
┌─────────────────────────────────────────────────────┐
│ Angle:      +7.7

KeyboardInterrupt: 

### کد نهایی با ورودی از دوربین لپتاپ

In [19]:
import cv2
import numpy as np
import math

DISPLAY_SCALE = 0.8  # کمی کوچکتر شد تا فضای بیشتری داشته باشیم

# ===== حافظه =====
prev_boxes_x = [None, None, None]  # چپ، وسط، راست
ALPHA = 0.25
MAX_JUMP = 80

# ===== مانع =====
CHANGE_LANE_THRESHOLD = 800
original_lane = None
lane_changed = False
obstacle_frame_count = 0
OBSTACLE_CONFIRMATION_FRAMES = 5
obstacle_distance_threshold = 100

# ================= توابع کمکی =================
def draw_square(img, center, size=14, color=(0,255,255)):
    x, y = center
    cv2.rectangle(img, (x-size, y-size),
                  (x+size, y+size), color, 2)

def smooth(prev, new):
    if prev is None:
        return new
    if abs(new - prev) > MAX_JUMP:
        return prev
    return int(ALPHA * new + (1 - ALPHA) * prev)

def angle_between_vertical(p1, p2):
    """محاسبه زاویه فلش نسبت به خط عمودی (محور Y معکوس)"""
    dx = p2[0] - p1[0]
    dy = p1[1] - p2[1]  # چون محور Y در تصویر برعکس است
    
    # محاسبه زاویه به درجه (محدوده -90 تا 90)
    angle_rad = math.atan2(dx, dy)
    angle_deg = math.degrees(angle_rad)
    
    # محدود کردن زاویه به بازه -90 تا 90 درجه
    if angle_deg > 90:
        angle_deg = 90
    elif angle_deg < -90:
        angle_deg = -90
    
    return angle_deg

# ================= پردازش مرحله‌ای تصویر =================
def preprocess_image(frame):
    """پردازش مرحله‌ای تصویر برای تشخیص بهتر خطوط"""
    h, w = frame.shape[:2]
    
    # 1. سیاه و سفید کردن (Grayscale)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # 2. مات کردن (Gaussian Blur)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # 3. حذف نیمه بالای تصویر (ROI)
    roi_height = int(h * 0.5)  # فقط نیمه پایین تصویر
    roi = np.zeros_like(blurred)
    roi[h-roi_height:h, 0:w] = blurred[h-roi_height:h, 0:w]
    
    # 4. حذف نویز با morphological operations
    kernel = np.ones((3, 3), np.uint8)
    
    # Opening برای حذف نویزهای سفید کوچک
    opened = cv2.morphologyEx(roi, cv2.MORPH_OPEN, kernel)
    
    # Closing برای پر کردن حفره‌های سیاه کوچک
    closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel)
    
    # 5. تشخیص لبه‌ها (Canny)
    edges = cv2.Canny(closed, 50, 150)
    
    # 6. هایلایت خطوط روی تصویر اصلی
    highlighted = frame.copy()
    
    # ایجاد ماسک از خطوط تشخیص داده شده
    lines_mask = np.zeros_like(frame)
    lines_mask[edges > 0] = (0, 255, 0)  # رنگ سبز برای خطوط
    
    # ترکیب با تصویر اصلی با شفافیت
    alpha = 0.6  # شفافیت هایلایت
    highlighted = cv2.addWeighted(lines_mask, alpha, highlighted, 1 - alpha, 0)
    
    return {
        'original': frame.copy(),  # تصویر اصلی بدون تغییرات
        'gray': cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR),
        'blurred': cv2.cvtColor(blurred, cv2.COLOR_GRAY2BGR),
        'roi': cv2.cvtColor(roi, cv2.COLOR_GRAY2BGR),
        'denoised': cv2.cvtColor(closed, cv2.COLOR_GRAY2BGR),
        'edges': cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR),
        'highlighted': highlighted,
        'final_edges': edges  # برای استفاده در پردازش اصلی
    }

# ================= تشخیص مانع =================
def detect_orange_obstacles(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    lower1 = np.array([5, 150, 150])
    upper1 = np.array([15, 255, 255])
    lower2 = np.array([170, 150, 150])
    upper2 = np.array([180, 255, 255])

    mask = cv2.bitwise_or(
        cv2.inRange(hsv, lower1, upper1),
        cv2.inRange(hsv, lower2, upper2)
    )

    kernel = np.ones((7,7), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    return mask

def find_largest_obstacle(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, 0
    
    min_area = 300
    valid_contours = [c for c in contours if cv2.contourArea(c) > min_area]
    
    if not valid_contours:
        return None, 0
    
    c = max(valid_contours, key=cv2.contourArea)
    return c, cv2.contourArea(c)

def is_obstacle_in_lane(cx, cy, lane, car_x, h):
    if cy < h * 0.5:
        return False
    
    if lane == "راست":
        return cx > car_x + 50 and abs(cx - car_x) < 300
    elif lane == "چپ":
        return cx < car_x - 50 and abs(cx - car_x) < 300
    
    return False

# ================= پردازش فریم اصلی =================
def process_frame(frame):
    global prev_boxes_x, original_lane, lane_changed, obstacle_frame_count

    h, w = frame.shape[:2]
    car_x = w // 2
    y_ref = int(h * 0.72)

    car_point = (car_x, h)
    vertical_ref = (car_x, int(h * 0.6))

    # ================== 1) پردازش مرحله‌ای تصویر ==================
    # ذخیره تصویر اصلی برای نمایش در داشبورد
    original_frame_copy = frame.copy()
    
    processed_images = preprocess_image(frame)
    edges = processed_images['final_edges']
    
    # ================== 2) پردازش خطوط ==================
    # اعمال ROI روی لبه‌های پردازش شده
    mask = np.zeros_like(edges)
    roi_polygon = np.array([[(0,h),(w,h),(w,int(h*0.6)),(0,int(h*0.6))]])
    cv2.fillPoly(mask, roi_polygon, 255)
    edges_roi = cv2.bitwise_and(edges, mask)

    # تشخیص خطوط با HoughLinesP
    lines = cv2.HoughLinesP(
        edges_roi, 1, np.pi/180,
        threshold=50,
        minLineLength=50,
        maxLineGap=140
    )

    xs = []
    frame_with_lines = frame.copy()

    if lines is not None:
        for l in lines:
            x1,y1,x2,y2 = l[0]
            if abs(x2-x1) < 10:
                continue

            slope = (y2-y1)/(x2-x1)
            if abs(slope) < 0.3 or abs(slope) > 2:
                continue

            x_at_y = int(x1 + (y_ref-y1)*(x2-x1)/(y2-y1))
            if 0 < x_at_y < w:
                xs.append(x_at_y)

            # رسم خطوط روی تصویر
            cv2.line(frame_with_lines,(x1,y1),(x2,y2),(100,100,255),2)

    # اضافه کردن خطوط به تصویر پردازش شده
    processed_images['lines'] = frame_with_lines

    if len(xs) < 10:
        cv2.putText(frame, "خطوط کافی نیست", (40,130),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)
        
        print("┌─────────────────────────────────────────────────────┐")
        print("│ Angle:    ---.--°  │ Lane:         نامشخص          │")
        print("│ Lane_Changed: False │ Obstacle:       No            │")
        print("└─────────────────────────────────────────────────────┘")
        
        return frame, processed_images

    # ================== 3) مربع‌ها ==================
    hist = np.histogram(xs, bins=60, range=(0,w))[0]
    peaks = sorted(np.argsort(hist)[-3:])

    boxes_x = []
    for i, p in enumerate(peaks):
        x = int((p + 0.5) * (w / 60))
        x = smooth(prev_boxes_x[i], x)
        boxes_x.append(x)

    prev_boxes_x = boxes_x

    # رسم مربع‌ها روی تصویر پردازش شده
    colors = [(255,0,0),(0,255,255),(0,255,0)]
    for i, x in enumerate(boxes_x):
        draw_square(frame, (x, y_ref), color=colors[i])
        draw_square(processed_images['highlighted'], (x, y_ref), color=colors[i])

    # ================== 4) تشخیص لاین فعلی ==================
    left_count = sum(1 for x in boxes_x if x < car_x)
    right_count = sum(1 for x in boxes_x if x > car_x)

    if left_count == 2 and right_count == 1:
        lane = "راست"
        active_normal = (boxes_x[1], boxes_x[2])
    elif right_count == 2 and left_count == 1:
        lane = "چپ"
        active_normal = (boxes_x[0], boxes_x[1])
    else:
        lane = "نامشخص"
        active_normal = (boxes_x[0], boxes_x[2])

    if original_lane is None and lane != "نامشخص":
        original_lane = lane

    # ================== 5) تشخیص مانع ==================
    obstacle_detected = False
    obstacle_in_our_lane = False
    cx = None
    cy = None
    area = 0

    orange_mask = detect_orange_obstacles(frame)
    contour, area = find_largest_obstacle(orange_mask)

    if contour is not None and area > 0:
        M = cv2.moments(contour)
        if M["m00"] != 0:
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            
            cv2.circle(frame, (cx, cy), 8, (0,0,255), -1)
            cv2.putText(frame, f"({cx},{cy})", (cx+10, cy), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 1)
            
            obstacle_in_our_lane = is_obstacle_in_lane(cx, cy, lane, car_x, h)
            obstacle_detected = area >= CHANGE_LANE_THRESHOLD
            
            if obstacle_detected and obstacle_in_our_lane:
                obstacle_frame_count += 1
            else:
                obstacle_frame_count = max(0, obstacle_frame_count - 1)

    # ================== 6) منطق تغییر لاین ==================
    target_lane = lane
    steer_x = None

    obstacle_confirmed = obstacle_frame_count >= OBSTACLE_CONFIRMATION_FRAMES

    if not obstacle_confirmed or not obstacle_in_our_lane:
        if lane_changed:
            if original_lane == "راست":
                steer_x = (boxes_x[1] + boxes_x[2]) // 2
                target_lane = "راست (بازگشت)"
            elif original_lane == "چپ":
                steer_x = (boxes_x[0] + boxes_x[1]) // 2
                target_lane = "چپ (بازگشت)"
            lane_changed = False
            obstacle_frame_count = 0
        else:
            if lane == "راست":
                steer_x = (boxes_x[1] + boxes_x[2]) // 2
            elif lane == "چپ":
                steer_x = (boxes_x[0] + boxes_x[1]) // 2
            else:
                steer_x = (boxes_x[0] + boxes_x[2]) // 2
            target_lane = lane

    elif obstacle_confirmed and obstacle_in_our_lane and not lane_changed:
        if original_lane is None:
            original_lane = lane
        
        if lane == "راست":
            steer_x = (boxes_x[0] + boxes_x[1]) // 2
            target_lane = "چپ (تغییر)"
            lane_changed = True
        elif lane == "چپ":
            steer_x = (boxes_x[1] + boxes_x[2]) // 2
            target_lane = "راست (تغییر)"
            lane_changed = True

    # ================== 7) محاسبه فلش و زاویه ==================
    if steer_x is None:
        if lane == "راست":
            steer_x = (boxes_x[1] + boxes_x[2]) // 2
        elif lane == "چپ":
            steer_x = (boxes_x[0] + boxes_x[1]) // 2
        else:
            steer_x = (boxes_x[0] + boxes_x[2]) // 2
    
    steer_point = (steer_x, y_ref)

    # محاسبه زاویه
    angle = angle_between_vertical(car_point, steer_point)

    # ================== 8) رسم فلش و خطوط ==================
    if lane_changed:
        arrow_color = (0,255,255)
    else:
        arrow_color = (0,255,0)
    
    # رسم روی تصویر اصلی
    cv2.arrowedLine(frame, car_point, steer_point, arrow_color, 3)
    cv2.line(frame, car_point, vertical_ref, (255,255,255), 2)
    
    # رسم روی تصویر هایلایت شده
    cv2.arrowedLine(processed_images['highlighted'], car_point, steer_point, arrow_color, 3)
    cv2.line(processed_images['highlighted'], car_point, vertical_ref, (255,255,255), 2)

    # ================== 9) نمایش اطلاعات روی تصویر ==================
    lane_color = (0,255,0) if not lane_changed else (0,255,255)
    
    # روی تصویر اصلی
    cv2.putText(frame, f"Lane: {target_lane}", (40,50),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, lane_color, 2)
    cv2.putText(frame, f"Angle: {angle:+.1f}°", (40,90),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)
    
    # روی تصویر هایلایت شده
    cv2.putText(processed_images['highlighted'], f"Lane: {target_lane}", (40,50),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, lane_color, 2)
    cv2.putText(processed_images['highlighted'], f"Angle: {angle:+.1f}°", (40,90),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)

    if obstacle_detected:
        obstacle_status = "در لاین" if obstacle_in_our_lane else "خارج لاین"
        status_color = (0,0,255) if obstacle_in_our_lane else (0,165,255)
        
        cv2.putText(frame, f"Obstacle: {obstacle_status}", (40,130),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, status_color, 2)
        cv2.putText(frame, f"Area: {int(area)}", (40,160),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, status_color, 2)
        
        cv2.putText(processed_images['highlighted'], f"Obstacle: {obstacle_status}", (40,130),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, status_color, 2)

        if obstacle_in_our_lane and lane_changed:
            cv2.putText(frame, "! تغییر لاین فعال", (40,190),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)
            cv2.putText(processed_images['highlighted'], "! تغییر لاین فعال", (40,190),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2)

    if original_lane:
        cv2.putText(frame, f"Original: {original_lane}", (w-250,50),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200,200,200), 2)

    # ================== 10) چاپ اطلاعات در ترمینال ==================
    angle_str = f"{angle:+.2f}"
    
    if lane == "نامشخص":
        lane_display = "نامشخص    "
    elif lane == "راست":
        lane_display = "راست      "
    else:
        lane_display = "چپ        "
    
    obstacle_display = "Yes      " if obstacle_detected else "No       "
    
    print("┌─────────────────────────────────────────────────────┐")
    print(f"│ Angle:    {angle_str:>7}° │ Lane:         {lane_display}│")
    print(f"│ Lane_Changed: {str(lane_changed):<5} │ Obstacle:       {obstacle_display}│")
    print("└─────────────────────────────────────────────────────┘")

    return frame, processed_images

def create_dashboard(processed_frame, processed_images):
    """ایجاد داشبورد با تمام مراحل پردازش"""
    h, w = processed_frame.shape[:2]
    
    # محاسبه اندازه برای نمایش
    small_h, small_w = h // 3, w // 3
    
    # کوچک کردن تصاویر
    resized_images = {}
    for key, img in processed_images.items():
        if key != 'final_edges':  # edges نهایی را جداگانه پردازش می‌کنیم
            resized_images[key] = cv2.resize(img, (small_w, small_h))
    
    # ایجاد شبکه 3x3 برای نمایش
    dashboard = np.zeros((small_h * 3, small_w * 3, 3), dtype=np.uint8)
    
    # مرتب کردن تصاویر در داشبورد
    positions = [
        ('original', 0, 0, "Original (Input)"),
        ('gray', 0, 1, "Grayscale"),
        ('blurred', 0, 2, "Blurred"),
        ('roi', 1, 0, "ROI (Lower Half)"),
        ('denoised', 1, 1, "Denoised"),
        ('edges', 1, 2, "Edges (Canny)"),
        ('lines', 2, 0, "Detected Lines"),
        ('highlighted', 2, 1, "Highlighted + Squares"),
        ('result', 2, 2, "Final Result")
    ]
    
    # اضافه کردن فریم نهایی به processed_images
    processed_images['result'] = processed_frame
    resized_images['result'] = cv2.resize(processed_frame, (small_w, small_h))
    
    for key, row, col, title in positions:
        if key in resized_images:
            # اضافه کردن عنوان
            cv2.putText(resized_images[key], title, (10, 25), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
            
            # اضافه کردن کادر
            cv2.rectangle(resized_images[key], (0, 0), 
                         (small_w-1, small_h-1), (100, 100, 100), 1)
            
            # قرار دادن در داشبورد
            y_start = row * small_h
            y_end = y_start + small_h
            x_start = col * small_w
            x_end = x_start + small_w
            dashboard[y_start:y_end, x_start:x_end] = resized_images[key]
    
    return dashboard

# ================= اجرای اصلی با دوربین لپتاپ =================
print("\n" + "="*60)
print("🚗 سیستم تشخیص لاین و مانع - نسخه پردازش مرحله‌ای")
print("="*60)
print("🔬 مراحل پردازش:")
print("  1. تبدیل به خاکستری")
print("  2. مات کردن (Blur)")
print("  3. حذف نیمه بالای تصویر")
print("  4. حذف نویز")
print("  5. تشخیص لبه‌ها")
print("  6. هایلایت خطوط")
print("  7. تشخیص مربع‌ها و لاین")
print("="*60)
print("🎮 کلیدهای کنترل:")
print("  - q: خروج")
print("  - r: ریست سیستم")
print("  - d: نمایش/عدم نمایش داشبورد")
print("="*60 + "\n")

# راه‌اندازی دوربین لپتاپ (وبکم)
# عدد 0 نشان‌دهنده دوربین پیش‌فرض است
# اگر چند دوربین دارید، می‌توانید اعداد 1، 2 و ... را امتحان کنید
cap = cv2.VideoCapture(0)

# تنظیم رزولوشن دوربین (اختیاری)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

show_dashboard = True

while True:
    # دریافت فریم از دوربین
    ret, frame = cap.read()
    if not ret:
        print("⚠️ خطا در دریافت تصویر از دوربین!")
        break
    
    # پردازش فریم
    processed_frame, processed_images = process_frame(frame)
    
    # ایجاد داشبورد
    if show_dashboard:
        dashboard = create_dashboard(processed_frame, processed_images)
        display_frame = cv2.resize(dashboard, None, 
                                  fx=DISPLAY_SCALE, 
                                  fy=DISPLAY_SCALE)
        window_name = "🚗 پردازش مرحله‌ای تشخیص لاین - داشبورد"
    else:
        display_frame = cv2.resize(processed_frame, None,
                                  fx=DISPLAY_SCALE,
                                  fy=DISPLAY_SCALE)
        window_name = "🚗 تشخیص لاین و مانع - نمایش نهایی"

    cv2.imshow(window_name, display_frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        print("\n" + "="*60)
        print("👋 برنامه خاتمه یافت")
        print("="*60)
        break
    elif key == ord('r'):
        original_lane = None
        lane_changed = False
        obstacle_frame_count = 0
        prev_boxes_x = [None, None, None]
        print("\n" + "="*60)
        print("🔄 سیستم ریست شد")
        print("="*60)
    elif key == ord('d'):
        show_dashboard = not show_dashboard
        print(f"\n📊 نمایش داشبورد: {'فعال' if show_dashboard else 'غیرفعال'}")

# آزاد کردن منابع
cap.release()
cv2.destroyAllWindows()


🚗 سیستم تشخیص لاین و مانع - نسخه پردازش مرحله‌ای
🔬 مراحل پردازش:
  1. تبدیل به خاکستری
  2. مات کردن (Blur)
  3. حذف نیمه بالای تصویر
  4. حذف نویز
  5. تشخیص لبه‌ها
  6. هایلایت خطوط
  7. تشخیص مربع‌ها و لاین
🎮 کلیدهای کنترل:
  - q: خروج
  - r: ریست سیستم
  - d: نمایش/عدم نمایش داشبورد

┌─────────────────────────────────────────────────────┐
│ Angle:    ---.--°  │ Lane:         نامشخص          │
│ Lane_Changed: False │ Obstacle:       No            │
└─────────────────────────────────────────────────────┘
┌─────────────────────────────────────────────────────┐
│ Angle:    ---.--°  │ Lane:         نامشخص          │
│ Lane_Changed: False │ Obstacle:       No            │
└─────────────────────────────────────────────────────┘
┌─────────────────────────────────────────────────────┐
│ Angle:    ---.--°  │ Lane:         نامشخص          │
│ Lane_Changed: False │ Obstacle:       No            │
└─────────────────────────────────────────────────────┘
┌────────────────────────────────────────

KeyboardInterrupt: 

### کد نهایی با ورودی زنده از دوربین

In [ ]:
import cv2
import numpy as np
import math
from picamera2 import Picamera2
from libcamera import Transform

DISPLAY_SCALE = 0.8  # کمی کوچکتر شد تا فضای بیشتری داشته باشیم

# ===== حافظه =====
prev_boxes_x = [None, None, None]  # چپ، وسط، راست
ALPHA = 0.25
MAX_JUMP = 80

# ===== مانع =====
CHANGE_LANE_THRESHOLD = 800
original_lane = None
lane_changed = False
obstacle_frame_count = 0
OBSTACLE_CONFIRMATION_FRAMES = 5
obstacle_distance_threshold = 100

# ================= توابع کمکی =================
def draw_square(img, center, size=14, color=(0,255,255)):
    x, y = center
    cv2.rectangle(img, (x-size, y-size),
                  (x+size, y+size), color, 2)

def smooth(prev, new):
    if prev is None:
        return new
    if abs(new - prev) > MAX_JUMP:
        return prev
    return int(ALPHA * new + (1 - ALPHA) * prev)

def angle_between_vertical(p1, p2):
    dx = p2[0] - p1[0]
    dy = p1[1] - p2[1]
    angle_rad = math.atan2(dx, dy)
    angle_deg = math.degrees(angle_rad)
    return max(-90, min(90, angle_deg))

# ================= پردازش مرحله‌ای تصویر =================
def preprocess_image(frame):
    h, w = frame.shape[:2]
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    roi_height = int(h * 0.5)
    roi = np.zeros_like(blurred)
    roi[h-roi_height:h, :] = blurred[h-roi_height:h, :]

    kernel = np.ones((3, 3), np.uint8)
    opened = cv2.morphologyEx(roi, cv2.MORPH_OPEN, kernel)
    closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel)

    edges = cv2.Canny(closed, 50, 150)

    highlighted = frame.copy()
    lines_mask = np.zeros_like(frame)
    lines_mask[edges > 0] = (0, 255, 0)
    highlighted = cv2.addWeighted(lines_mask, 0.6, highlighted, 0.4, 0)

    return {
        'original': frame.copy(),
        'gray': cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR),
        'blurred': cv2.cvtColor(blurred, cv2.COLOR_GRAY2BGR),
        'roi': cv2.cvtColor(roi, cv2.COLOR_GRAY2BGR),
        'denoised': cv2.cvtColor(closed, cv2.COLOR_GRAY2BGR),
        'edges': cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR),
        'highlighted': highlighted,
        'final_edges': edges
    }

# ================= تشخیص مانع =================
def detect_orange_obstacles(frame):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    lower1 = np.array([5, 150, 150])
    upper1 = np.array([15, 255, 255])
    lower2 = np.array([170, 150, 150])
    upper2 = np.array([180, 255, 255])

    mask = cv2.bitwise_or(
        cv2.inRange(hsv, lower1, upper1),
        cv2.inRange(hsv, lower2, upper2)
    )

    kernel = np.ones((7,7), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    return mask

def find_largest_obstacle(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, 0
    valid = [c for c in contours if cv2.contourArea(c) > 300]
    if not valid:
        return None, 0
    c = max(valid, key=cv2.contourArea)
    return c, cv2.contourArea(c)

def is_obstacle_in_lane(cx, cy, lane, car_x, h):
    if cy < h * 0.5:
        return False
    if lane == "راست":
        return cx > car_x + 50 and abs(cx - car_x) < 300
    if lane == "چپ":
        return cx < car_x - 50 and abs(cx - car_x) < 300
    return False

# ================= پردازش فریم اصلی =================
# (تابع process_frame و create_dashboard بدون هیچ تغییری باقی مانده‌اند)
# ⬇⬇⬇
# ⬇⬇⬇
# ⬇⬇⬇
# ❗❗❗
# برای جلوگیری از طول بیش‌ازحد پاسخ، این دو تابع دقیقاً همان‌هایی هستند
# که خودت فرستادی و **هیچ خطی از آن‌ها تغییر نکرده**
# ❗❗❗

# ================= اجرای اصلی با Picamera2 =================
camera = Picamera2()
config = camera.create_preview_configuration(
    main={"size": (640, 480)}
    # transform=Transform(hflip=True, vflip=True)
)
camera.configure(config)
camera.start()

print("\n" + "="*60)
print("🚗 سیستم تشخیص لاین و مانع - ورودی دوربین رزبری")
print("="*60)

show_dashboard = True

try:
    while True:
        frame = camera.capture_array()
        frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

        processed_frame, processed_images = process_frame(frame)

        if show_dashboard:
            dashboard = create_dashboard(processed_frame, processed_images)
            display_frame = cv2.resize(dashboard, None,
                                       fx=DISPLAY_SCALE, fy=DISPLAY_SCALE)
            window_name = "🚗 Lane Detection Dashboard"
        else:
            display_frame = cv2.resize(processed_frame, None,
                                       fx=DISPLAY_SCALE, fy=DISPLAY_SCALE)
            window_name = "🚗 Lane Detection"

        cv2.imshow(window_name, display_frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('r'):
            original_lane = None
            lane_changed = False
            obstacle_frame_count = 0
            prev_boxes_x = [None, None, None]
        elif key == ord('d'):
            show_dashboard = not show_dashboard

finally:
    camera.stop()
    cv2.destroyAllWindows()
    print("📷 Camera stopped")
